**Authors:** Pablo Rodríguez Elvira, Adrián Segura Onorato

# Data Processing

In this notebook, we describe the complete data cleaning and preprocessing pipeline used to transform the raw dataset into a clean version ready for visualization.

We processed the data programmatically using **pandas**, which makes the workflow fully reproducible. Starting from the raw file `simpsons_episodes.csv`, we applied a sequence of cleaning, transformation, and validation steps, and then exported the final clean dataset as `simpsons_episodes_clean.csv`.

The goal of this process is to ensure that the data used for the visualizations is consistent, well-structured, and free from issues that could negatively affect the analysis.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

input_file = Path("simpsons_episodes.csv")
output_file = Path("simpsons_episodes_clean.csv")

df = pd.read_csv(input_file)
df.head()

,id,image_url,imdb_rating,imdb_votes,number_in_season,number_in_series,original_air_date,original_air_year,production_code,season,title,us_viewers_in_millions,video_url,views
0,10,http://static-media.fxx.com/img/FX_Networks_-_...,7.4,1511.0,10,10,1990-03-25,1990,7G10,1,Homer's Night Out,30.3,http://www.simpsonsworld.com/video/275197507879,50816.0
1,12,http://static-media.fxx.com/img/FX_Networks_-_...,8.3,1716.0,12,12,1990-04-29,1990,7G12,1,Krusty Gets Busted,30.4,http://www.simpsonsworld.com/video/288019523914,62561.0
2,14,http://static-media.fxx.com/img/FX_Networks_-_...,8.2,1638.0,1,14,1990-10-11,1990,7F03,2,"Bart Gets an ""F""",33.6,http://www.simpsonsworld.com/video/260539459671,59575.0
3,17,http://static-media.fxx.com/img/FX_Networks_-_...,8.1,1457.0,4,17,1990-11-01,1990,7F01,2,Two Cars in Every Garage and Three Eyes on Eve...,26.1,http://www.simpsonsworld.com/video/260537411822,64959.0
4,19,http://static-media.fxx.com/img/FX_Networks_-_...,8.0,1366.0,6,19,1990-11-15,1990,7F08,2,Dead Putting Society,25.4,http://www.simpsonsworld.com/video/260539459670,50691.0


## Cleaning Steps 
The cleaning process consists of the following steps:


### **1. Column selection**  
We kept only the variables relevant for the analysis and visualization: `title`, `season`, `number_in_season`, `original_air_date`, `imdb_rating`, `imdb_votes`, and `us_viewers_in_millions`.

   These variables provide the essential information needed to identify episodes, analyze their chronological distribution, compare audience size, and study episode ratings. The other columns were removed because they were redundant, derived from other variables, or not relevant to the goals of the project. For instance, `original_air_year` can be obtained from `original_air_date`, while `id`, `production_code`, `image_url`, `video_url`, and `views` were not necessary for our visual analysis.

In [2]:
important_cols = [
    "title",
    "season",
    "number_in_season",
    "number_in_series",
    "original_air_date",
    "imdb_rating",
    "imdb_votes",
    "us_viewers_in_millions",
]

df = df[important_cols].copy()
df.head()

,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions
0,Homer's Night Out,1,10,10,1990-03-25,7.4,1511.0,30.3
1,Krusty Gets Busted,1,12,12,1990-04-29,8.3,1716.0,30.4
2,"Bart Gets an ""F""",2,1,14,1990-10-11,8.2,1638.0,33.6
3,Two Cars in Every Garage and Three Eyes on Eve...,2,4,17,1990-11-01,8.1,1457.0,26.1
4,Dead Putting Society,2,6,19,1990-11-15,8.0,1366.0,25.4


### **2. Type conversion**  
   We converted the `original_air_date` column to datetime format using `pd.to_datetime()` in order to work with dates correctly and extract temporal information later.

In [3]:
print("Types before conversion:")
print(df.dtypes)

df["original_air_date"] = pd.to_datetime(df["original_air_date"], errors="coerce")

print("\nTypes after conversion:")
print(df.dtypes)

Types before conversion:
title                      object
season                      int64
number_in_season            int64
number_in_series            int64
original_air_date          object
imdb_rating               float64
imdb_votes                float64
us_viewers_in_millions    float64
dtype: object

Types after conversion:
title                             object
season                             int64
number_in_season                   int64
number_in_series                   int64
original_air_date         datetime64[ns]
imdb_rating                      float64
imdb_votes                       float64
us_viewers_in_millions           float64
dtype: object


### **3. Missing value inspection**  
   We checked the dataset for missing values in all selected columns to identify incomplete records that could affect the visualizations.

In [4]:
print("NA's per column:")
print(df.isna().sum())

print("Rows with at least one NA value:")
df[df.isna().any(axis=1)]

NA's per column:
title                     0
season                    0
number_in_season          0
number_in_series          0
original_air_date         0
imdb_rating               3
imdb_votes                3
us_viewers_in_millions    6
dtype: int64
Rows with at least one NA value:


,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions
59,Lisa's Date with Density,8,7,160,1996-12-15,7.8,1005.0,NaN
65,The Canine Mutiny,8,20,173,1997-04-13,7.7,913.0,NaN
234,"Friends and Family""[203]",28,2,598,2016-10-02,NaN,NaN,NaN
235,"The Town""[205]",28,3,599,2016-10-09,NaN,NaN,NaN
236,"Treehouse of Horror XXVII""[207]",28,4,600,2016-10-16,NaN,NaN,NaN
320,Hurricane Neddy,8,8,161,1996-12-29,8.8,1268.0,NaN


### **4. Filtering and imputation**  
   We removed the records from season 28 and interpolated missing values in `us_viewers_in_millions` within each season. This allowed us to preserve the complete seasonal structure of the data while filling gaps in a consistent way.

In [5]:
df = df[df["season"] != 28].copy()

df["us_viewers_in_millions"] = (
    df.groupby("season")["us_viewers_in_millions"]
    .transform(lambda s: s.interpolate())
)

df.head()

,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions
0,Homer's Night Out,1,10,10,1990-03-25,7.4,1511.0,30.3
1,Krusty Gets Busted,1,12,12,1990-04-29,8.3,1716.0,30.4
2,"Bart Gets an ""F""",2,1,14,1990-10-11,8.2,1638.0,33.6
3,Two Cars in Every Garage and Three Eyes on Eve...,2,4,17,1990-11-01,8.1,1457.0,26.1
4,Dead Putting Society,2,6,19,1990-11-15,8.0,1366.0,25.4


### **5. Duplicate check**  
   We verified whether duplicated rows existed in the cleaned subset of the dataset.

In [6]:
duplicates = df[df.duplicated()]
print("Duplicates found:", len(duplicates))
duplicates

Duplicates found: 0


,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions


### **6. Feature engineering**  
   We generated several derived variables from the original columns in order to support the visual analysis:
   - `air_year`: year in which the episode was released
   - `air_month`: month in which the episode was released
   - `weekday_num`: numeric representation of the weekday
   - `weekday`: name of the weekday
   - `season_episode`: combined season and episode code in the format `SxxExx`
   - `rating_group`: categorical grouping of IMDb ratings into intervals

In [7]:
# Extract date-related columns
df["air_year"] = df["original_air_date"].dt.year
df["air_month"] = df["original_air_date"].dt.month
df["weekday_num"] = df["original_air_date"].dt.weekday

weekday_map = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

df["weekday"] = df["weekday_num"].map(weekday_map)

# Combine season and episode number into a single identifier
df["season_episode"] = (
    "S"
    + df["season"].astype(int).astype(str).str.zfill(2)
    + "E"
    + df["number_in_season"].astype(int).astype(str).str.zfill(2)
)

# Create rating groups for easier visualization
df["rating_group"] = np.select(
    [
        df["imdb_rating"] < 5,
        (df["imdb_rating"] >= 5) & (df["imdb_rating"] < 6),
        (df["imdb_rating"] >= 6) & (df["imdb_rating"] < 7),
        (df["imdb_rating"] >= 7) & (df["imdb_rating"] < 8),
        (df["imdb_rating"] >= 8) & (df["imdb_rating"] < 9),
        df["imdb_rating"] >= 9
    ],
    [
        "< 5",
        "5.0 - 5.9",
        "6.0 - 6.9",
        "7.0 - 7.9",
        "8.0 - 8.9",
        "9.0 - 10.0"
    ],
    default="Unknown"
)

df.head()

,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions,air_year,air_month,weekday_num,weekday,season_episode,rating_group
0,Homer's Night Out,1,10,10,1990-03-25,7.4,1511.0,30.3,1990,3,6,Sunday,S01E10,7.0 - 7.9
1,Krusty Gets Busted,1,12,12,1990-04-29,8.3,1716.0,30.4,1990,4,6,Sunday,S01E12,8.0 - 8.9
2,"Bart Gets an ""F""",2,1,14,1990-10-11,8.2,1638.0,33.6,1990,10,3,Thursday,S02E01,8.0 - 8.9
3,Two Cars in Every Garage and Three Eyes on Eve...,2,4,17,1990-11-01,8.1,1457.0,26.1,1990,11,3,Thursday,S02E04,8.0 - 8.9
4,Dead Putting Society,2,6,19,1990-11-15,8.0,1366.0,25.4,1990,11,3,Thursday,S02E06,8.0 - 8.9


### **7. Ordering**  
   Finally, we sorted the dataset by air date, season, and episode number to keep the records in chronological order.

In [8]:
df = df.sort_values(["original_air_date", "season", "number_in_season"]).reset_index(drop=True)

df

,title,season,number_in_season,number_in_series,original_air_date,imdb_rating,imdb_votes,us_viewers_in_millions,air_year,air_month,weekday_num,weekday,season_episode,rating_group
0,Simpsons Roasting on an Open Fire,1,1,1,1989-12-17,8.2,3734.0,26.70,1989,12,6,Sunday,S01E01,8.0 - 8.9
1,Bart the Genius,1,2,2,1990-01-14,7.8,1973.0,24.50,1990,1,6,Sunday,S01E02,7.0 - 7.9
2,Homer's Odyssey,1,3,3,1990-01-21,7.5,1709.0,27.50,1990,1,6,Sunday,S01E03,7.0 - 7.9
3,There's No Disgrace Like Home,1,4,4,1990-01-28,7.8,1701.0,20.20,1990,1,6,Sunday,S01E04,7.0 - 7.9
4,Bart the General,1,5,5,1990-02-04,8.1,1732.0,27.10,1990,2,6,Sunday,S01E05,8.0 - 8.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,How Lisa Got Her Marge Back,27,18,592,2016-04-10,6.4,228.0,2.55,2016,4,6,Sunday,S27E18,6.0 - 6.9
592,Fland Canyon,27,19,593,2016-04-24,7.1,229.0,2.77,2016,4,6,Sunday,S27E19,7.0 - 7.9
593,To Courier with Love,27,20,594,2016-05-08,6.7,193.0,2.52,2016,5,6,Sunday,S27E20,6.0 - 6.9
594,Simprovised,27,21,595,2016-05-15,6.4,227.0,2.80,2016,5,6,Sunday,S27E21,6.0 - 6.9


### **8. Export**  
   The final cleaned dataset was exported as `simpsons_episodes_clean.csv`.

In [9]:
df.to_csv(output_file, index=False)
print(f"File saved in: {output_file}")

File saved in: simpsons_episodes_clean.csv
